## Running the Experiment

This notebook is designed to be executed sequentially, cell by cell, from top to bottom. Each step depends on the outputs of the previous ones, so skipping or reordering cells may lead to errors or inconsistent results.

The entire pipeline is organized into clearly defined stages, including environment setup, data loading, preprocessing, model training, evaluation, and comparison with baseline models.

When executed in order, the notebook performs the following steps:

### 1. Environment Setup

Installs the required libraries for model training, dataset handling, and evaluation. This step needs to be executed once.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn


### 2. Imports and Reproducibility

All required libraries are imported, and a fixed random seed is established to ensure reproducibility across runs. The computation device is automatically selected (GPU if available, otherwise CPU).

In [ ]:
import torch
import random
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

dispozitiv = torch.device("cuda" if torch.cuda.is_available() else "cpu")
samanta = 42
random.seed(samanta)
np.random.seed(samanta)
torch.manual_seed(samanta)
if dispozitiv.type == "cuda":
    torch.cuda.manual_seed_all(samanta)


### 3. Model and Tokenizer Initialization

Loads the pre-trained CodeBERT model (`microsoft/codebert-base`) and its corresponding tokenizer.  
The model is configured for binary sequence classification (2 labels) and moved to the selected device.

Note: Variable names are in Romanian (e.g., `nume_model`, `tokenizator`), but they correspond to standard Hugging Face components.

In [ ]:
nume_model = "microsoft/codebert-base"
tokenizator = AutoTokenizer.from_pretrained(nume_model)
model = AutoModelForSequenceClassification.from_pretrained(nume_model, num_labels=2)
model.to(dispozitiv)


### 4. Dataset Loading

The SemEval-2026 Task 13 dataset (Subtask A) is loaded using the Hugging Face `datasets` library.  
The predefined training and validation splits are used for all experiments.

In [ ]:
dataset_initial = load_dataset("DaniilOr/SemEval-2026-Task13", "A")
date_antrenare = dataset_initial["train"]
date_validare = dataset_initial["validation"]



### 5. Data Preprocessing and Tokenization

Code snippets are tokenized using the pre-trained tokenizer, with truncation and padding applied to a fixed maximum length of 256 tokens.

The datasets are processed using this function, unnecessary columns are removed, and the data is converted into PyTorch tensors (`input_ids`, `attention_mask`, `labels`) for training.

Note: Variable names are in Romanian but follow standard Hugging Face preprocessing conventions.

In [ ]:
lungime_max = 256
def tokenizeaza(exemple):
    texte = exemple["code"]
    lbl = exemple["label"]
    t = tokenizator(
        texte,
        truncation=True,
        padding="max_length",
        max_length=lungime_max
    )
    t["labels"] = lbl
    return t

train_proc = date_antrenare.map(tokenizeaza, batched=True, remove_columns=["code", "generator", "language"])
valid_proc = date_validare.map(tokenizeaza, batched=True, remove_columns=["code", "generator", "language"])
train_proc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
valid_proc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


### 6. Data Loaders

PyTorch data loaders are created for efficient batch processing.

The training loader uses shuffling to improve generalization, while the validation loader remains deterministic to ensure consistent evaluation.

In [ ]:
marime_batch = 32

loader_train = DataLoader(train_proc, batch_size=marime_batch, shuffle=True)
loader_valid = DataLoader(valid_proc, batch_size=marime_batch, shuffle=False)




### 7. Layer Freezing Strategy

Implements a gradual unfreezing strategy for the Transformer backbone.

- All parameters in the `model.roberta` module are initially frozen (`requires_grad = False`).
- The helper function `deblocheaza_straturi(n)` progressively unfreezes the last `n` encoder layers.
- The value of `n` is clipped between `0` and the total number of encoder layers to avoid invalid indexing.

This strategy enables controlled fine-tuning, where deeper encoder layers are progressively unfrozen during training.

In [ ]:
for p in model.roberta.parameters():
    p.requires_grad = False
straturi = list(model.roberta.encoder.layer)
total = len(straturi)
pas_deblocare = 4
def deblocheaza_straturi(n):
    n = max(0, min(total, n))
    start = total - n
    for i in range(total):
        strat = straturi[i]
        permite = (i >= start)
        for p in strat.parameters():
            p.requires_grad = permite


### 8. Optimization Setup

Applies the gradual unfreezing strategy and configures the optimizer and learning rate scheduler.

- The last `pas_deblocare` encoder layers are unfrozen using `deblocheaza_straturi(...)`.
- Model parameters are divided into two groups:
  - **Encoder parameters** (lower learning rate)
  - **Classifier parameters** (higher learning rate)

Different learning rates are used:
- `lr_enc = 2e-5`
- `lr_clf = 1e-4`

The `AdamW` optimizer is initialized with these parameter groups.

A linear learning rate scheduler with warmup is also configured:
- Warmup steps = 6% of total training steps  
- Total training steps = number of batches × number of epochs

This setup stabilizes training and allows faster adaptation of the classification head.

In [ ]:
deblocheaza_straturi(pas_deblocare)
lista_enc = []
lista_clf = []
for nume, p in model.named_parameters():
    if "classifier" in nume:
        lista_clf.append(p)
    else:
        lista_enc.append(p)
lr_enc = 2e-5
lr_clf = 1e-4
nr_epoci = 3
opt = AdamW(
    [
        {"params": [x for x in lista_enc if x.requires_grad], "lr": lr_enc},
        {"params": [x for x in lista_clf if x.requires_grad], "lr": lr_clf},
    ]
)
pasi_epoca = len(loader_train)
total_pasi = pasi_epoca * nr_epoci
warm = int(total_pasi * 0.06)
scheduler = get_linear_schedule_with_warmup(
    opt,
    num_warmup_steps=warm,
    num_training_steps=total_pasi
)


### 9. Evaluation Function

Defines the evaluation procedure on the validation set.

The model is set to evaluation mode (`model.eval()`), and predictions are computed without gradient tracking (`torch.no_grad()`).

For each batch:
- Inputs and labels are moved to the selected device
- Logits are computed and converted to predicted labels using `argmax`

Predictions and ground truth labels are aggregated across all batches.

Performance is measured using the **macro-averaged F1 score**.

In [ ]:
def testeaza():
    model.eval()
    toate_pred = []
    toate_adev = []
    with torch.no_grad():
        for lot in loader_valid:
            intrari = lot["input_ids"].to(dispozitiv)
            masca = lot["attention_mask"].to(dispozitiv)
            etic = lot["labels"].to(dispozitiv)
            ies = model(input_ids=intrari, attention_mask=masca)
            scoruri = ies.logits
            pred_lot = torch.argmax(scoruri, dim=1)
            toate_pred += pred_lot.cpu().numpy().tolist()
            toate_adev += etic.cpu().numpy().tolist()

    f1_macro = f1_score(toate_adev, toate_pred, average="macro")
    return f1_macro


### 10. Storage Setup

Configures storage for saving model checkpoints and training artifacts.

- Google Drive is mounted at `/content/drive`
- A directory is created at `/content/drive/MyDrive/licenta`

This directory is used to store:
- Model checkpoints
- Training artifacts

Note: This step assumes execution in Google Colab with access to Google Drive. If running in a different environment, the storage path should be updated accordingly.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs("/content/drive/MyDrive/licenta", exist_ok=True)

### 11. Checkpoint Loading

Loads a training checkpoint if available and restores the training state.

If a checkpoint is found:
- Model weights are loaded
- Optimizer and scheduler states are restored
- The best validation score (`cel_mai_bun`) is recovered
- Training resumes from the next epoch

If no checkpoint is found:
- Training starts from scratch (epoch 1)

The checkpoint path must be correctly configured to enable reproducibility and training resumption.

In [ ]:
import os
import torch

fisier_model = "codebert_finetuned_binar.pt"
cale_checkpoint = "/content/drive/MyDrive/licenta/checkpoint_codebert_taskA.pt"
cel_mai_bun = 0.0
start_epoca = 0
if os.path.exists(cale_checkpoint):
    checkpoint = torch.load(cale_checkpoint, map_location=dispozitiv)
    model.load_state_dict(checkpoint["model_state"])
    opt.load_state_dict(checkpoint["opt_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    cel_mai_bun = checkpoint["cel_mai_bun"]
    start_epoca = checkpoint["epoca"] + 1
    print("Am gasit checkpoint. Reiau de la epoca:", start_epoca + 1)
else:
    print("Nu am gasit checkpoint. Porneasc de la epoca 1.")


### 12. Training and Checkpointing

Runs the training loop across multiple epochs with gradual unfreezing.

- The number of unfrozen encoder layers increases progressively with each epoch
- Each training step includes a forward pass, loss computation, backward pass, gradient clipping, and optimizer + scheduler updates

After each epoch:
- The model is evaluated on the validation set using the macro-averaged F1 score
- The best-performing model (highest validation F1) is saved

A checkpoint is saved after each epoch, allowing training to be resumed later if needed.

In [ ]:

for ep in range(start_epoca, nr_epoci):
    nr_deblocate = (ep + 1) * pas_deblocare
    deblocheaza_straturi(nr_deblocate)
    model.train()
    lista_pierderi = []
    for lot in loader_train:
        id_uri = lot["input_ids"].to(dispozitiv)
        masca_at = lot["attention_mask"].to(dispozitiv)
        lab = lot["labels"].to(dispozitiv)
        ies = model(input_ids=id_uri, attention_mask=masca_at, labels=lab)
        loss_curent = ies.loss
        lista_pierderi.append(loss_curent.item())
        loss_curent.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        scheduler.step()
        opt.zero_grad()
    scor_f1 = testeaza()
    print("Epoca:", ep + 1,
          "pierdere_medie:", float(np.mean(lista_pierderi)),
          "F1_macro_validare:", scor_f1)
    if scor_f1 > cel_mai_bun:
        cel_mai_bun = scor_f1
        torch.save(model.state_dict(), fisier_model)
        print("Model imbunatatit. F1_macro_validare:", cel_mai_bun)
    checkpoint = {
        "epoca": ep,
        "model_state": model.state_dict(),
        "opt_state": opt.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "cel_mai_bun": cel_mai_bun,
    }
    torch.save(checkpoint, cale_checkpoint)
    print("Checkpoint salvat dupa epoca", ep + 1)


### 13. Expected Output (Illustrative Example)

During training, the model prints progress information at each epoch.

An example output is shown below (messages are displayed in Romanian, as defined in the implementation):

Epoca: 1 pierdere_medie: 0.06501832012975728 F1_macro_validare: 0.9896813795293021
Model imbunatatit. F1_macro_validare: 0.9896813795293021
Checkpoint salvat dupa epoca 1

Epoca: 2 pierdere_medie: 0.030396034302691232 F1_macro_validare: 0.9917943512624884
Model imbunatatit. F1_macro_validare: 0.9917943512624884
Checkpoint salvat dupa epoca 2

Epoca: 3 pierdere_medie: 0.024541230915440245 F1_macro_validare: 0.9927752983672286
Model imbunatatit. F1_macro_validare: 0.9927752983672286
Checkpoint salvat dupa epoca 3


### 14. Validation Subset Selection

Creates a reproducible subset of up to 500 validation examples for lightweight evaluation and analysis.

- The validation set is shuffled using a fixed seed (`seed=42`)
- The first 500 samples are selected


In [ ]:
from datasets import Dataset
nr_exemple_mici = 500
valid_amestecat = date_validare.shuffle(seed=42)
nr_exemple_mici = min(nr_exemple_mici, len(valid_amestecat))
valid_mic = valid_amestecat.select(range(nr_exemple_mici))
print(len(valid_mic))
print(valid_mic[0])

### 15. Subset Preprocessing and Data Loader

Prepares the validation subset for model evaluation.

- Applies tokenization using the predefined tokenizer
- Removes unused columns (`code`, `generator`, `language`)
- Formats the dataset as PyTorch tensors (`input_ids`, `attention_mask`, `labels`)

A DataLoader is created to enable efficient batch processing during evaluation.

In [ ]:

from torch.utils.data import DataLoader
valid_mic_proc = valid_mic.map(
    tokenizeaza,
    batched=True,
    remove_columns=["code", "generator", "language"]
)
valid_mic_proc.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)
batch_mic = 32
loader_valid_mic = DataLoader(
    valid_mic_proc,
    batch_size=batch_mic,
    shuffle=False
)


### 16. Evaluation on Subset

Evaluates the model on the validation subset.

- The model is set to evaluation mode (`model.eval()`)
- Predictions are computed without gradient tracking (`torch.no_grad()`), reducing memory usage and improving inference efficiency
- Logits are converted to class predictions using `argmax`

Performance is measured using the macro-averaged F1 score.

In [ ]:
from sklearn.metrics import f1_score
import torch
def testeaza_loader(loader_valid_curent):
    model.eval()
    toate_pred = []
    toate_adev = []
    with torch.no_grad():
        for lot in loader_valid_curent:
            intrari = lot["input_ids"].to(dispozitiv)
            masca  = lot["attention_mask"].to(dispozitiv)
            etic   = lot["labels"].to(dispozitiv)

            ies = model(input_ids=intrari, attention_mask=masca)
            scoruri = ies.logits
            pred_lot = torch.argmax(scoruri, dim=1)

            toate_pred += pred_lot.cpu().numpy().tolist()
            toate_adev += etic.cpu().numpy().tolist()
    f1_macro = f1_score(toate_adev, toate_pred, average="macro")
    return f1_macro


### 17. Evaluation of Fine-Tuned Model

Loads the fine-tuned model and evaluates it on the validation subset.

- Model weights are loaded from the saved checkpoint
- Evaluation is performed using the previously defined function

The resulting macro F1 score provides a quick estimate of model performance on a smaller validation sample.

In [ ]:
cale_model_finetuned = fisier_model
state_dict = torch.load(cale_model_finetuned, map_location=dispozitiv)
model.load_state_dict(state_dict)
model.to(dispozitiv)
f1_mic_codebert = testeaza_loader(loader_valid_mic)
print("F1 macro pe mini-validare (500 exemple) CodeBERT:", f1_mic_codebert)


### 18. Subset Evaluation Result

On the sampled validation subset (500 examples), the fine-tuned CodeBERT model achieved a macro F1 score of approximately 0.99.


### 19. Data Extraction for Analysis

Code snippets and corresponding labels are extracted from the validation subset.

The sizes of the extracted arrays are verified to ensure data consistency.

In [ ]:
fragmente_cod = valid_mic["code"]
etichete_adev = valid_mic["label"]
len(fragmente_cod), len(etichete_adev)

### 20. LLM Baseline Evaluation (DeepSeek Coder)

An open-source LLM (DeepSeek Coder) is evaluated as a baseline for the classification task.

- The model is prompted to classify code snippets as:
  - `1` → machine-generated 
  - `0` → human-written 
- Predictions are generated for each sample in the validation subset
- The macro F1 score is computed to compare performance with the fine-tuned model

This provides a reference point for assessing the effectiveness of the fine-tuned CodeBERT model.
The model is prompted in Romanian to ensure clear and constrained responses.

In [ ]:
!pip install -q transformers accelerate

from sklearn.metrics import f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "deepseek-ai/deepseek-coder-1.3b-instruct"
tokenizer_llm = AutoTokenizer.from_pretrained(model_name)
model_llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)
task_txt = """
Ești un clasificator. Primești un fragment de cod și trebuie să spui:

1 -> dacă este probabil generat de un model AI
0 -> dacă este probabil scris de un om

Răspunde strict cu un singur caracter: 0 sau 1.
"""
def clasifica_llm(fragment):
    prompt = task_txt + "\n\nFragment:\n```code\n" + fragment + "\n```\n\nEticheta:"
    x = tokenizer_llm(prompt, return_tensors="pt").to(model_llm.device)
    out = model_llm.generate(
        **x,
        max_new_tokens=4,
        do_sample=False,
        pad_token_id=tokenizer_llm.eos_token_id
    )
    txt = tokenizer_llm.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if "0" in txt and "1" not in txt:
        return 0
    if "1" in txt and "0" not in txt:
        return 1
    try:
        return int(txt[0])
    except:
        return 0
fragmente = valid_mic["code"]
etichete = valid_mic["label"]
predictii = []
for i, cod in enumerate(fragmente):
    p = clasifica_llm(cod)
    predictii.append(p)

    if (i + 1) % 50 == 0:
        print("Procesate", i + 1, "fragmente")
f1_llm = f1_score(etichete, predictii, average="macro")
print("F1 pe mini-validare (LLM open-source):", f1_llm)


### 21. LLM Baseline Result

On the sampled validation subset (500 examples), the open-source LLM (DeepSeek Coder) achieved a macro F1 score of **0.3007**.

This result highlights the limitations of zero-shot LLMs for fine-grained code classification tasks and emphasizes the importance of task-specific fine-tuning.

### 22. LLM Baseline Evaluation (Qwen 2.5 Coder)

An open-source LLM (Qwen 2.5 Coder) is evaluated as an additional baseline for the classification task.

- The model is prompted to classify code snippets as:
  - `1` → machine-generated code
  - `0` → human-written code
- Predictions are generated for each sample in the validation subset
- The macro F1 score is computed for comparison with the fine-tuned CodeBERT model

The model is prompted in Romanian to ensure clear and constrained responses.


In [ ]:
!pip install -q transformers accelerate
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

tok = AutoTokenizer.from_pretrained(model_name)
mod = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

if tok.pad_token_id is None:
    tok.pad_token_id = tok.eos_token_id

task_txt = """
Ești un clasificator. Primești un fragment de cod și trebuie să spui:

1 -> dacă este probabil generat de un model AI
0 -> dacă este probabil scris de un om

Răspunde strict cu un singur caracter: 0 sau 1.
"""

def clasifica(cod):
    chat = [
        {"role": "system", "content": task_txt},
        {"role": "user", "content": f"{cod}\n\nEticheta:"}
    ]
    text = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    x = tok(text, return_tensors="pt").to(mod.device)
    y = mod.generate(**x, max_new_tokens=4, do_sample=False, pad_token_id=tok.pad_token_id)
    r = tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    if "0" in r and "1" not in r:
        return 0
    if "1" in r and "0" not in r:
        return 1
    try:
        return int(r[0])
    except:
        return 0

from sklearn.metrics import f1_score

coduri = valid_mic["code"]
lab = valid_mic["label"]
pred = []
for i, c in enumerate(coduri):
    pred.append(clasifica(c))
    if (i + 1) % 50 == 0:
        print(i + 1, "/", len(coduri))
print("F1:", f1_score(lab, pred, average="macro"))



### 23. Qwen Baseline Result

On the sampled validation subset (500 examples), Qwen 2.5 Coder achieved a macro F1 score of **0.4109**.

This result is significantly lower than the fine-tuned CodeBERT model, but slightly higher than the DeepSeek baseline, highlighting the limitations of zero-shot LLM classification for this task.

### 24. Export Validation Subset to CSV

The sampled validation subset (code snippets and labels) is exported to a CSV file.

The exported file can be used for:
- manual inspection
- external analysis
- ensuring reproducibility outside the notebook

In [ ]:
import pandas as pd
fisier_csv = "/content/drive/MyDrive/licenta/fragmente_random_500.csv"

df = pd.DataFrame({
    "code": valid_mic["code"],
    "label": valid_mic["label"]
})
df.to_csv(fisier_csv, index=False)
print("salvat in:", fisier_csv)


### (Optional) Save Model in HuggingFace Format

The fine-tuned model and tokenizer are saved in HuggingFace format for easier reuse and deployment.

This step is not required for training (as checkpoints are already used), but it simplifies loading the model for inference and sharing.

In [ ]:
cale_salveaza = "/content/drive/MyDrive/licenta/codebert_taskA/model_finetuned"
model.save_pretrained(cale_salveaza)
tokenizator.save_pretrained(cale_salveaza)


### (Optional) Save Tokenized Dataset

The tokenized dataset is stored using the HuggingFace `DatasetDict` format.

This enables efficient reuse without repeating preprocessing steps and supports reproducibility across experiments.

In [ ]:
from datasets import DatasetDict
set_final = DatasetDict({
    "train": train_proc,
    "validation": valid_proc
})
set_final.save_to_disk("/content/drive/MyDrive/licenta/dataset_tokenizat_taskA")


### (Optional) Model Architecture Summary

This cell displays a summary of the model architecture using `torchinfo`, including input/output dimensions and the number of parameters.

This step is optional and is intended for inspection and better understanding of the model structure.

In [ ]:
!pip install torchinfo

In [ ]:
from torchinfo import summary

summary(
    model,
    input_size=(1, lungime_max),
    dtypes=[torch.long],
    col_names=["input_size", "output_size", "num_params", "trainable"],
)